# RCQ Qwen3.6 FP Smoke Notebook

FP-only remote GPU smoke for `Qwen/Qwen3.6-35B-A3B`. This notebook does not quantize or save model weights. It writes small diagnostics under `/kaggle/working/outputs/qwen36_fp_smoke_notebook`.

In [ ]:
import gc
import json
import os
import platform
import shlex
import subprocess
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any
from urllib.parse import urlparse, urlunparse

WORKING = Path('/kaggle/working')
OUTPUT_DIR = WORKING / 'outputs' / 'qwen36_fp_smoke_notebook'
COMPETITION_DIR = Path('/kaggle/input/nvidia-nemotron-model-reasoning-challenge')
DEFAULT_MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'


def set_secret_env(names: list[str]) -> None:
    try:
        from kaggle_secrets import UserSecretsClient
    except Exception:
        return
    secrets = UserSecretsClient()
    for name in names:
        if os.environ.get(name):
            continue
        try:
            value = secrets.get_secret(name)
        except Exception:
            value = None
        if value:
            os.environ[name] = value


set_secret_env([
    'HF_TOKEN',
    'HUGGING_FACE_HUB_TOKEN',
    'RCQ_REPO_URL',
    'RCQ_COMMIT_SHA',
    'RCQ_GIT_TOKEN',
])

os.environ.setdefault('HF_HOME', str(WORKING / 'hf_cache'))
os.environ.setdefault('TRANSFORMERS_CACHE', str(WORKING / 'hf_cache'))
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def run(argv: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None, display_argv: list[str] | None = None) -> None:
    shown = display_argv if display_argv is not None else argv
    print('$', ' '.join(shlex.quote(part) for part in shown), flush=True)
    subprocess.run(argv, cwd=cwd, env=env, check=True)


print(f'competition_dir_exists={COMPETITION_DIR.exists()}')
print(f'output_dir={OUTPUT_DIR}')
print(f"has_hf_token={bool(os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN'))}")
print(f"has_rcq_repo_url={bool(os.environ.get('RCQ_REPO_URL'))}")
print(f"has_rcq_commit_sha={bool(os.environ.get('RCQ_COMMIT_SHA'))}")
run(['nvidia-smi'])

In [ ]:
if os.environ.get('RCQ_SKIP_PIP_INSTALL') != '1':
    run([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '-U',
        'transformers',
        'accelerate',
        'safetensors',
        'tokenizers',
        'huggingface-hub',
    ])
else:
    print('RCQ_SKIP_PIP_INSTALL=1; using Kaggle image packages')

In [ ]:
def repo_url_with_optional_token(repo_url: str) -> tuple[str, str]:
    token = os.environ.get('RCQ_GIT_TOKEN')
    if not token:
        return repo_url, repo_url
    parsed = urlparse(repo_url)
    if parsed.scheme != 'https' or not parsed.netloc:
        raise RuntimeError('RCQ_GIT_TOKEN is only supported with an https RCQ_REPO_URL.')
    authed = urlunparse(parsed._replace(netloc=f'x-access-token:{token}@{parsed.netloc}'))
    redacted = urlunparse(parsed._replace(netloc=f'x-access-token:***@{parsed.netloc}'))
    return authed, redacted


def run_repo_script_if_requested() -> bool:
    if os.environ.get('RCQ_USE_REPO_RUNNER') != '1':
        return False
    repo_url = os.environ.get('RCQ_REPO_URL')
    if not repo_url:
        raise RuntimeError('RCQ_USE_REPO_RUNNER=1 requires RCQ_REPO_URL as a Kaggle secret/env var.')
    checkout = WORKING / 'rcq_moe_impl'
    if checkout.exists():
        run(['rm', '-rf', str(checkout)])
    clone_url, display_clone_url = repo_url_with_optional_token(repo_url)
    run(
        ['git', 'clone', '--depth', '1', clone_url, str(checkout)],
        display_argv=['git', 'clone', '--depth', '1', display_clone_url, str(checkout)],
    )
    commit_sha = os.environ.get('RCQ_COMMIT_SHA')
    if commit_sha:
        run(['git', 'fetch', '--depth', '1', 'origin', commit_sha], cwd=checkout)
        run(['git', 'checkout', '--detach', commit_sha], cwd=checkout)
    run([sys.executable, '-m', 'pip', 'install', '-e', '.'], cwd=checkout)
    script_args = [
        sys.executable,
        'scripts/qwen36_fp_smoke.py',
        '--model-id', os.environ.get('QWEN36_MODEL_ID', DEFAULT_MODEL_ID),
        '--output-dir', str(OUTPUT_DIR),
        '--sequence-length', os.environ.get('RCQ_FP_SMOKE_SEQUENCE_LENGTH', '256'),
        '--eval-batch-size', os.environ.get('RCQ_FP_SMOKE_EVAL_BATCH_SIZE', '1'),
        '--eval-batches', os.environ.get('RCQ_FP_SMOKE_EVAL_BATCHES', '1'),
    ]
    if os.environ.get('QWEN36_TRUST_REMOTE_CODE') == '1':
        script_args.append('--trust-remote-code')
    run(script_args, cwd=checkout)
    return True


repo_runner_used = run_repo_script_if_requested()
print(f'repo_runner_used={repo_runner_used}')

In [ ]:
if not repo_runner_used:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    @dataclass
    class ModuleSummary:
        name: str
        class_name: str
        child_names: list[str]
        parameter_count: int

    def write_json(path: Path, payload: dict[str, Any]) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')

    def run_command(argv: list[str]) -> str:
        try:
            result = subprocess.run(argv, check=False, capture_output=True, text=True)
        except FileNotFoundError:
            return f'{argv[0]} not found'
        output = (result.stdout + result.stderr).strip()
        return output if output else f'{argv[0]} exited with code {result.returncode}'

    def gpu_snapshot() -> dict[str, Any]:
        if not torch.cuda.is_available():
            return {'cuda_available': False, 'device_count': 0}
        devices = []
        for idx in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(idx)
            devices.append({
                'index': idx,
                'name': props.name,
                'total_memory_gb': props.total_memory / 1024**3,
                'major': props.major,
                'minor': props.minor,
            })
        return {
            'cuda_available': True,
            'device_count': torch.cuda.device_count(),
            'devices': devices,
            'max_memory_allocated_gb': torch.cuda.max_memory_allocated() / 1024**3,
            'max_memory_reserved_gb': torch.cuda.max_memory_reserved() / 1024**3,
        }

    def torch_dtype(name: str):
        mapping = {
            'auto': 'auto',
            'float32': torch.float32,
            'float16': torch.float16,
            'bfloat16': torch.bfloat16,
        }
        if name not in mapping:
            raise ValueError(f'unsupported dtype {name!r}')
        return mapping[name]

    def synthetic_fixture_documents() -> list[str]:
        return [
            'Mixture of experts routing selects sparse expert MLPs for each token.',
            'Router coherent quantization preserves shared expert subspaces before residual quantization.',
            'Calibration text should exercise code, math, chat, and general language distributions.',
            'This FP-only smoke checks model loading, sparse module structure, and one tiny loss pass.',
        ]

    def make_eval_batch(tokenizer, *, batch_size: int, sequence_length: int) -> tuple[torch.Tensor, torch.Tensor, int]:
        docs = synthetic_fixture_documents()
        repeats = (batch_size + len(docs) - 1) // len(docs)
        texts = (docs * repeats)[:batch_size]
        encoded = tokenizer(
            texts,
            padding='max_length',
            truncation=True,
            max_length=sequence_length,
            return_tensors='pt',
        )
        return encoded['input_ids'], encoded['attention_mask'], len(docs)

    def summarize_moe_modules(model: torch.nn.Module, *, max_modules: int) -> list[ModuleSummary]:
        summaries: list[ModuleSummary] = []
        markers = ('moe', 'expert', 'router', 'gate')
        for name, module in model.named_modules():
            class_name = module.__class__.__name__
            haystack = f'{name.lower()} {class_name.lower()}'
            if not any(marker in haystack for marker in markers):
                continue
            summaries.append(ModuleSummary(
                name=name,
                class_name=class_name,
                child_names=list(module._modules.keys()),
                parameter_count=int(sum(p.numel() for p in module.parameters(recurse=False))),
            ))
            if len(summaries) >= max_modules:
                break
        return summaries

    def next_token_loss(logits: torch.Tensor, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> float:
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = input_ids[:, 1:].contiguous()
        shift_mask = attention_mask[:, 1:].contiguous().bool()
        losses = torch.nn.functional.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)).float(),
            shift_labels.view(-1),
            reduction='none',
        ).view_as(shift_labels)
        denom = shift_mask.sum().clamp_min(1)
        return float((losses * shift_mask).sum().div(denom).item())

    started = time.time()
    model_id = os.environ.get('QWEN36_MODEL_ID', DEFAULT_MODEL_ID)
    revision = os.environ.get('QWEN36_MODEL_REVISION') or None
    sequence_length = int(os.environ.get('RCQ_FP_SMOKE_SEQUENCE_LENGTH', '256'))
    eval_batch_size = int(os.environ.get('RCQ_FP_SMOKE_EVAL_BATCH_SIZE', '1'))
    dtype_name = os.environ.get('RCQ_FP_SMOKE_DTYPE', 'bfloat16')
    device_map = os.environ.get('RCQ_FP_SMOKE_DEVICE_MAP', 'auto')
    trust_remote_code = os.environ.get('QWEN36_TRUST_REMOTE_CODE') == '1'
    local_files_only = os.environ.get('QWEN36_LOCAL_FILES_ONLY') == '1'
    token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')

    env_payload = {
        'mode': 'embedded_notebook_fp_smoke',
        'python': sys.version,
        'platform': platform.platform(),
        'torch': torch.__version__,
        'cuda': gpu_snapshot(),
        'nvidia_smi': run_command(['nvidia-smi']),
        'model_id': model_id,
        'revision': revision,
        'hf_home': os.environ.get('HF_HOME'),
        'transformers_cache': os.environ.get('TRANSFORMERS_CACHE'),
        'competition_dir_exists': COMPETITION_DIR.exists(),
    }
    write_json(OUTPUT_DIR / 'metadata.json', env_payload)

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        revision=revision,
        token=token,
        trust_remote_code=trust_remote_code,
        local_files_only=local_files_only,
    )
    input_ids_cpu, attention_mask_cpu, eval_docs = make_eval_batch(
        tokenizer,
        batch_size=eval_batch_size,
        sequence_length=sequence_length,
    )

    load_started = time.time()
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        revision=revision,
        token=token,
        torch_dtype=torch_dtype(dtype_name),
        device_map=device_map,
        trust_remote_code=trust_remote_code,
        local_files_only=local_files_only,
        low_cpu_mem_usage=True,
    )
    model.eval()
    load_elapsed = time.time() - load_started

    module_summaries = summarize_moe_modules(
        model,
        max_modules=int(os.environ.get('RCQ_FP_SMOKE_MAX_MOE_MODULES', '200')),
    )
    module_lines = [
        f"{item.name}\t{item.class_name}\tparams_recurse_false={item.parameter_count}\tchildren={','.join(item.child_names)}"
        for item in module_summaries
    ]
    (OUTPUT_DIR / 'module_structure.txt').write_text('\n'.join(module_lines) + '\n', encoding='utf-8')

    device = next(model.parameters()).device
    input_ids = input_ids_cpu.to(device)
    attention_mask = attention_mask_cpu.to(device)

    forward_started = time.time()
    with torch.inference_mode():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
        logits = outputs.logits
        loss = next_token_loss(logits, input_ids, attention_mask)
    forward_elapsed = time.time() - forward_started

    metrics = {
        'model_id': model_id,
        'revision': revision,
        'model_class': model.__class__.__name__,
        'tokenizer_class': tokenizer.__class__.__name__,
        'text_source': 'embedded_synthetic_fixture_documents',
        'eval_docs': eval_docs,
        'eval_input_shape': list(input_ids_cpu.shape),
        'eval_attention_tokens': int(attention_mask_cpu.sum().item()),
        'logits_shape': list(logits.shape),
        'next_token_loss': loss,
        'load_elapsed_sec': load_elapsed,
        'forward_elapsed_sec': forward_elapsed,
        'elapsed_sec': time.time() - started,
        'moe_module_count_reported': len(module_summaries),
        'cuda_after_forward': gpu_snapshot(),
    }
    write_json(OUTPUT_DIR / 'fp_metrics.json', metrics)
    print(json.dumps(metrics, indent=2, sort_keys=True))

    del model, logits, outputs
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()